
# Optimización y Validación de Modelos en Scikit-Learn

En esta notebook se explican detalladamente las siguientes funciones de Scikit-Learn:

- `GridSearchCV`
- `RandomizedSearchCV`
- `cross_val_score`

Además:
- Se muestra cómo funcionan internamente.
- Se explican los parámetros más importantes.
- Se aplican a modelos de Machine Learning.
- Se comparan resultados.
- Se incluyen ejemplos completos con:
  - KNN
  - Árboles de Decisión




# 1. ¿Por qué optimizar hiperparámetros?

Cuando entrenamos un modelo de aprendizaje automático, existen dos tipos de valores:

## Parámetros
Son aprendidos automáticamente por el modelo.

Ejemplo:
- Los pesos de una regresión lineal.
- Las probabilidades en Naive Bayes o Regresión logística.

## Hiperparámetros
Son configurados manualmente antes del entrenamiento.

Ejemplos:
- Número de vecinos en KNN (`n_neighbors`)
- Profundidad máxima de un árbol (`max_depth`)
- Tipo de distancia (`metric`)
- Criterio de división (`criterion`)

Elegir malos hiperparámetros puede producir:
- Underfitting
- Overfitting
- Baja precisión

Por eso existen técnicas automáticas como:
- GridSearchCV
- RandomizedSearchCV



In [ ]:

# Importación de librerías

import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report



# 2. Carga del dataset

Usaremos el dataset Iris.

Contiene:
- Largo y ancho del sépalo
- Largo y ancho del pétalo
- Especie de flor

Es un problema de clasificación multiclase.


In [ ]:

iris = load_iris()

X = iris.data
y = iris.target

df = pd.DataFrame(X, columns=iris.feature_names)
df["target"] = y

df.head()


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0



# División de datos

Separaremos:
- Datos de entrenamiento
- Datos de prueba

El modelo aprenderá con entrenamiento y será evaluado con prueba.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (120, 4)
Prueba: (30, 4)



# 3. cross_val_score

## ¿Qué hace?

`cross_val_score` realiza validación cruzada.

En lugar de entrenar y evaluar una sola vez:
- divide los datos varias veces
- entrena múltiples modelos
- calcula métricas en cada división

Esto da una evaluación mucho más confiable.

---

# ¿Cómo funciona la validación cruzada?

Supongamos:
```python
cv=5
```

El dataset se divide en 5 partes:

- 4 partes para entrenar
- 1 parte para validar

Esto se repite 5 veces cambiando la parte de validación.

---

# Ventajas

- Reduce riesgo de overfitting
- Evaluación más estable
- Aprovecha mejor los datos

---

# Parámetros importantes

## estimator
Modelo que se quiere evaluar.

Ejemplo:
```python
KNeighborsClassifier()
```

## X
Variables predictoras.

## y
Variable objetivo.

## cv
Cantidad de particiones.

Ejemplo:
```python
cv=5
```

## scoring
Métrica de evaluación.

Ejemplos:
- accuracy
- precision
- recall
- f1

## n_jobs
Cantidad de núcleos del CPU.

```python
n_jobs=-1
```
usa todos los núcleos disponibles.



In [ ]:

modelo_knn = KNeighborsClassifier()

scores = cross_val_score(
    estimator=modelo_knn,
    X=X,
    y=y,
    cv=5,
    scoring="accuracy"
)

print("Scores de cada fold:")
print(scores)

print("\nAccuracy promedio:")
print(scores.mean())


Scores de cada fold:
[0.96666667 1.         0.93333333 0.96666667 1.        ]

Accuracy promedio:
0.9733333333333334



# 4. GridSearchCV

## ¿Qué hace?

`GridSearchCV` prueba TODAS las combinaciones posibles de hiperparámetros.

Ejemplo:

```python
n_neighbors = [3,5,7]
weights = ['uniform','distance']
```

Combinaciones:
- (3, uniform)
- (3, distance)
- (5, uniform)
- (5, distance)
- (7, uniform)
- (7, distance)

GridSearchCV entrenará un modelo para cada combinación.

---

# ¿Cómo funciona internamente?

Para cada combinación:

1. Entrena el modelo
2. Realiza validación cruzada
3. Calcula métricas
4. Guarda resultados
5. Selecciona la mejor combinación

---

# Ventajas

- Encuentra muy buenos parámetros
- Método exhaustivo
- Muy preciso

# Desventajas

- Puede ser lento
- Muchas combinaciones = mucho tiempo

---

# Parámetros importantes

## estimator
Modelo a optimizar.

## param_grid
Diccionario con hiperparámetros.

Ejemplo:

```python
{
    "n_neighbors": [3,5,7],
    "weights": ["uniform", "distance"]
}
```

## cv
Cantidad de folds.

## scoring
Métrica utilizada.

## verbose
Muestra el progreso.

## n_jobs
Cantidad de núcleos CPU.

## refit
Reentrena automáticamente el mejor modelo.



In [ ]:

parametros_knn = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=parametros_knn,
    cv=5,
    scoring=["accuracy", "recall"],
    refit='accuracy',  #
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Mejores parámetros:")
print(grid_search.best_params_)

print("\nMejor score:")
print(grid_search.best_score_)


Fitting 5 folds for each of 16 candidates, totalling 80 fits
Mejores parámetros:
{'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'uniform'}

Mejor score:
0.9583333333333334


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(



# Evaluación del mejor modelo encontrado

`best_estimator_` contiene automáticamente el mejor modelo.


In [ ]:

mejor_knn = grid_search.best_estimator_

predicciones = mejor_knn.predict(X_test)

print("Accuracy en test:")
print(accuracy_score(y_test, predicciones))

print("\nReporte de clasificación")
print(classification_report(y_test, predicciones))



# 5. RandomizedSearchCV

## ¿Qué hace?

`RandomizedSearchCV` NO prueba todas las combinaciones.

En cambio:
- elige combinaciones aleatorias
- prueba una cantidad limitada

---

# Diferencia con GridSearchCV

## GridSearchCV
Prueba TODAS las combinaciones.

## RandomizedSearchCV
Prueba combinaciones ALEATORIAS.

---

# Ventajas

- Mucho más rápido
- Escala mejor
- Ideal para muchos hiperparámetros

# Desventajas

- Puede no encontrar la combinación óptima
- Depende del azar

---

# Parámetros importantes

## estimator
Modelo.

## param_distributions
Diccionario de parámetros.

## n_iter
Cantidad de combinaciones aleatorias.

Ejemplo:
```python
n_iter=20
```

## cv
Cantidad de folds.

## scoring
Métrica.

## random_state
Controla aleatoriedad.

## n_jobs
Uso de CPU.



In [ ]:

parametros_tree = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, None],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 4, 6, 8, 10],
    "min_samples_leaf": [1, 2, 3, 4]
}

random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(),
    param_distributions=parametros_tree,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Mejores parámetros encontrados:")
print(random_search.best_params_)

print("\nMejor score:")
print(random_search.best_score_)



# Evaluación del mejor árbol encontrado


In [ ]:

mejor_tree = random_search.best_estimator_

predicciones_tree = mejor_tree.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, predicciones_tree))

print("\nReporte de clasificación")
print(classification_report(y_test, predicciones_tree))



# 6. Comparación entre métodos

| Método | Ventaja | Desventaja |
|---|---|---|
| cross_val_score | Evaluación confiable | No optimiza parámetros |
| GridSearchCV | Encuentra mejor combinación | Muy lento |
| RandomizedSearchCV | Más rápido | Puede no hallar el óptimo |

---

# ¿Cuándo usar cada uno?

## cross_val_score
Cuando:
- queremos evaluar estabilidad
- comparar modelos

## GridSearchCV
Cuando:
- tenemos pocos hiperparámetros
- buscamos máxima precisión

## RandomizedSearchCV
Cuando:
- hay MUCHOS parámetros
- el entrenamiento es costoso
- necesitamos velocidad




# 7. Parámetros frecuentes en modelos

## KNN

### n_neighbors
Cantidad de vecinos.

### weights
- uniform
- distance

### metric
Métrica de distancia.

Ejemplos:
- euclidean
- manhattan

---

## Árboles de Decisión

### max_depth
Profundidad máxima.

### criterion
Criterio de división:
- gini
- entropy

### min_samples_split
Mínimo de muestras para dividir.

### min_samples_leaf
Mínimo de muestras por hoja.




# 8. Conclusión

Estas funciones son fundamentales en Machine Learning:

- `cross_val_score`
  - evalúa modelos de forma robusta

- `GridSearchCV`
  - busca exhaustivamente los mejores hiperparámetros

- `RandomizedSearchCV`
  - optimiza más rápido usando búsqueda aleatoria

Son herramientas esenciales para:
- mejorar precisión
- reducir overfitting
- comparar modelos
- automatizar tuning de hiperparámetros
